In [11]:
# Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms.functional import normalize
import torchvision.models as tvmodels
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms 

import numpy as np
import rasterio

import os

from skimage.io import imread

In [13]:
pre_datafolder= r'C:\Users\b1114674\Desktop\thesis\pre_text_data\pre'
post_datafolder = r'C:\Users\b1114674\Desktop\thesis\pre_text_data\post'
    
pre_pathlist = [os.path.join(pre_datafolder,img) for img in os.listdir(pre_datafolder)]

post_pathlist= [os.path.join(post_datafolder,img) for img in os.listdir(post_datafolder)]

In [21]:
pre_meanstd = {"mean":[103.29,94.94, 97.53],"std":[48.63, 36.42, 31.22]}
post_meanstd = {"mean":[122.0, 110.35, 116.4],"std":[47.99, 33.89, 32.04]}

In [23]:
# Dataset and Dataloader

class CustomDataset(Dataset):

    def __init__(self,pre,post,pre_meanstd,post_meanstd):
        super().__init__()

        self.pre_meanstd = pre_meanstd
        self.post_meanstd = post_meanstd


        self.pre = pre
        self.post = post


    def __getitem__(self, index):
        pre = self.pre[index]
        post = self.post[index]

        pre = imread(pre)
        post = imread(post)

        pre, post = self.pre_transform(pre), self.post_transform(post)
        return pre, post


    def pre_transform(self,x):

        x = torch.from_numpy(x).permute(2,0,1).float()
        
        x = normalize(x,self.pre_meanstd['mean'],self.pre_meanstd['std'],inplace=True)

        return x

    def post_transform(self,x):

        x  = torch.from_numpy(x).permute(2,0,1).float()
        
        x = normalize(x,self.post_meanstd['mean'],self.post_meanstd['std'],inplace=True)

        return x
    
    
    def __len__(self):
        return len(self.pre)
    
dataset = CustomDataset(pre_pathlist,post_pathlist,pre_meanstd,post_meanstd)

train_loader = DataLoader(dataset,shuffle=True,batch_size=200)

In [ ]:
#Position Embeddings implementation


In [ ]:
#Random and Change masking



In [13]:
# Model
import timm.models.vision_transformer as Vit
# Pre trained VIT encoder

class VitEncoder(nn.Module):

    def __init__(self,embed_dim,attention_heads,depth=5):
        super().__init__()

        self.depth = depth

        self.encoder = Vit.Block(embed_dim,attention_heads,qkv_bias=True)

        self.projection = nn.Linear(embed_dim,embed_dim)

    def forward(self,x):

        for i in range (self.depth):
            x = self.encoder(x)

        x = self.projection(x)

        return x

In [ ]:
class CFMAE(nn.Module):

    def __init__ (self,encoder,embed_dim):
        super().__init__()

        self.encoder = encoder
        self.decoder = nn.Sequential(
            nn.Conv2d(
                in_channels=embed_dim,
                out_channels=3)
        )

    def forward(self,pre,post):
        pre = self.encoder(pre)
        post = self.encoder(post)
        zdiff = pre-post
        out = torch.concatenate[zdiff+pre]
        out = self.decoder(out)
        return out

In [ ]:
import timm
encoder = timm.create_model(model_name='vit_base_patch16_224',pretrained=True,drop_rate=0.2,num_classes = 0,global_pool= '')

In [ ]:
encoder

In [ ]:
class TemporalGeoMAE_SingleDecoder(nn.Module):
    """
    Temporal MAE with SINGLE decoder (post-only)
    More efficient for change detection downstream tasks
    """
    
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
        encoder_name: str = 'vit_base_patch16_224',
        encoder_embed_dim: int = 768,
        decoder_embed_dim: int = 512,
        decoder_depth: int = 8,
        decoder_num_heads: int = 16,
        mask_ratio: float = 0.75,
    ):
        super().__init__()
        
        self.num_patches = (img_size // patch_size) ** 2
        self.encoder_embed_dim = encoder_embed_dim
        
        # Shared encoder
        self.encoder = timm.create_model(
            encoder_name,
            pretrained=False,
            num_classes=0,
            global_pool='',
        )
        
        # Projections for latent alignment
        self.pre_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
        )
        
        self.post_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
        )
        
        # Change encoder (rich difference)
        self.change_encoder = nn.Sequential(
            nn.Linear(encoder_embed_dim * 4, encoder_embed_dim * 2),
            nn.GELU(),
            nn.Linear(encoder_embed_dim * 2, encoder_embed_dim),
        )
        
        # SINGLE decoder embedding
        # Takes concatenated [z_pre, z_diff]
        self.decoder_embed = nn.Linear(
            encoder_embed_dim * 2,  # z_pre + z_diff
            decoder_embed_dim
        )
        
        # Single mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        
        # Decoder position embeddings
        self.decoder_pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, decoder_embed_dim)
        )
        
        # SINGLE decoder (for post-image only)
        self.decoder = nn.ModuleList([
            Block(
                dim=decoder_embed_dim,
                num_heads=decoder_num_heads,
                mlp_ratio=4.0,
            )
            for _ in range(decoder_depth)
        ])
        
        # Single prediction head
        self.decoder_pred = nn.Linear(
            decoder_embed_dim,
            patch_size ** 2 * in_channels
        )
        
        # Optional: Change detection head
        self.change_classifier = nn.Sequential(
            nn.Linear(encoder_embed_dim, encoder_embed_dim // 2),
            nn.GELU(),
            nn.Linear(encoder_embed_dim // 2, 2),
        )
        
        self.initialize_weights()
    
    def forward(self, img_pre, img_post):
        # Encode both images
        z_pre_enc, mask_pre, ids_restore_pre = self.encode(img_pre, self.mask_ratio)
        z_post_enc, mask_post, ids_restore_post = self.encode(img_post, self.mask_ratio)
        
        # Project to aligned space
        z_pre = self.pre_projection(z_pre_enc)
        z_post = self.post_projection(z_post_enc)
        
        # Compute change encoding
        z_diff = self.compute_change_encoding(z_pre, z_post)
        
        # KEY DIFFERENCE: Concatenate z_pre and z_diff for decoder
        # This forces decoder to use BOTH context and change info
        z_combined = torch.cat([z_pre, z_diff], dim=-1)  # (B, N_visible, 2*D)
        z_dec = self.decoder_embed(z_combined)  # (B, N_visible, D_dec)
        
        # Decode ONLY post-image
        pred_post = self.decode(
            z_dec,
            ids_restore_post,
            self.mask_token,
            self.decoder,
            self.decoder_pred
        )
        
        # Change detection
        z_diff_pooled = z_diff.mean(dim=1)
        change_logits = self.change_classifier(z_diff_pooled)
        
        return {
            'pred_post': pred_post,
            'mask_post': mask_post,
            'change_logits': change_logits,
            'z_pre': z_pre,      # Save for analysis
            'z_post': z_post,
            'z_diff': z_diff,
        }
    
    def compute_loss(
        self,
        img_post: torch.Tensor,
        pred_post: torch.Tensor,
        mask_post: torch.Tensor,
        change_labels: Optional[torch.Tensor] = None,
    ):
        """
        Simplified loss: Only post-reconstruction + optional change classification
        """
        # Patchify target
        target_post = self.patchify(img_post)
        
        # MSE loss on masked patches
        loss_recon = (pred_post - target_post) ** 2
        loss_recon = loss_recon.mean(dim=-1)
        loss_recon = (loss_recon * mask_post).sum() / mask_post.sum()
        
        loss_dict = {'loss_recon': loss_recon}
        
        # Optional change classification
        if change_labels is not None:
            loss_change = F.cross_entropy(
                self.change_classifier(z_diff_pooled),
                change_labels
            )
            loss_dict['loss_change'] = loss_change
            loss_dict['loss_total'] = loss_recon + 0.1 * loss_change
        else:
            loss_dict['loss_total'] = loss_recon
        
        return loss_dict

In [ ]:
def guided_random_masking(
    pre_image: torch.Tensor,      # (B, C, H, W)
    post_image: torch.Tensor,     # (B, C, H, W)
    num_patches: int,              # Total patches
    mask_ratio: float = 0.75,      # Target masking ratio
    bias_strength: float = 0.3,    # How much to bias toward changes
    temperature: float = 1.0,      # Smoothness of change scores
):
    """
    Guided random masking with soft bias toward changed areas
    
    Args:
        bias_strength: 0.0 = pure random, 1.0 = fully change-guided
        temperature: Higher = smoother probabilities
    """
    B, C, H, W = pre_image.shape
    
    # Step 1: Compute change scores per patch
    # For simplicity, using pixel-level difference
    # In practice, you could use:
    # - Feature-level difference (better!)
    # - Pretrained change detector
    # - Multi-scale differences
    
    pixel_diff = torch.abs(pre_image - post_image)  # (B, C, H, W)
    
    # Average over channels and spatial locations within each patch
    patch_size = H // int(num_patches ** 0.5)
    
    # Reshape to patches
    patches_pre = rearrange(
        pre_image,
        'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size
    )
    patches_post = rearrange(
        post_image,
        'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
        p1=patch_size, p2=patch_size
    )
    
    # Compute change score per patch
    patch_diff = torch.abs(patches_pre - patches_post)  # (B, N, P*P*C)
    change_scores = patch_diff.mean(dim=-1)  # (B, N) - average difference per patch
    
    # Normalize to [0, 1]
    change_scores = change_scores / (change_scores.max(dim=1, keepdim=True)[0] + 1e-8)
    
    # Apply temperature for smoothing
    change_scores = torch.sigmoid((change_scores - 0.5) / temperature)
    
    # Step 2: Create biased mask probabilities
    base_prob = mask_ratio
    
    # Blend base probability with change scores
    mask_probs = (
        (1 - bias_strength) * base_prob +  # Base random component
        bias_strength * change_scores       # Change-guided component
    )
    
    # Step 3: Sample masks
    # Use Gumbel-Softmax for differentiable sampling (if needed)
    # Or simple top-k for discrete sampling
    
    num_masked = int(num_patches * mask_ratio)
    
    # Add noise for randomness
    noise = torch.rand(B, num_patches, device=pre_image.device)
    
    # Combine probabilities with noise
    sampling_scores = mask_probs + 0.5 * noise  # 50% random, 50% guided
    
    # Get top-k indices to mask
    _, ids_shuffle = torch.sort(sampling_scores, dim=1, descending=True)
    
    # First num_masked indices are masked
    mask = torch.zeros(B, num_patches, device=pre_image.device)
    mask.scatter_(1, ids_shuffle[:, :num_masked], 1)
    
    # Create ids_restore
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    
    return mask, ids_restore, change_scores


# Usage in your model:
def forward(self, img_pre, img_post):
    # Pre-image: Pure random masking
    z_pre, mask_pre, ids_restore_pre = self.encode(
        img_pre, 
        mask_ratio=0.75
    )
    
    # Post-image: Guided random masking
    mask_post, ids_restore_post, change_scores = guided_random_masking(
        img_pre, 
        img_post,
        num_patches=self.num_patches,
        mask_ratio=0.75,
        bias_strength=0.3,  # 30% bias toward changes
        temperature=1.0,
    )
    
    # Apply mask to post-image patches
    post_patches = self.encoder.patch_embed(img_post)
    len_keep = int(self.num_patches * (1 - 0.75))
    ids_keep = ids_restore_post[:, :len_keep]
    z_post = torch.gather(
        post_patches, 
        dim=1, 
        index=ids_keep.unsqueeze(-1).expand(-1, -1, post_patches.shape[-1])
    )